## Inheritance, MRO, and Object Internals


**Topics covered:** Single inheritance, extending/overriding, `super()`, cooperative multiple inheritance, mixins, attribute lookup & MRO, `isinstance` / `issubclass`, `__dict__`, `__slots__`, class dictionaries, composition over inheritance.


## 1. Single Inheritance

Inheritance lets one class reuse the attributes and methods of another.
- **Subclass / Child** — the class that inherits
- **Superclass / Parent** — the class being inherited from


In [ ]:
class Animal:
    def __init__(self, name, sound):
        self.name  = name
        self.sound = sound

    def speak(self):
        return f'{self.name} says {self.sound}!'

    def __repr__(self):
        return f'{type(self).__name__}(name={self.name!r})'


class Dog(Animal):
    # Dog inherits everything from Animal.
    def fetch(self):
        # New method — only Dog has this.
        return f'{self.name} fetches the ball!'


class Cat(Animal):
    def speak(self):
        # Override the parent's speak with a cat-specific version.
        return f'{self.name} says Meow softly.'


rex  = Dog('Rex',  'Woof')
luna = Cat('Luna', 'Meow')

print(rex.speak())    # inherited from Animal
print(rex.fetch())    # only on Dog
print(luna.speak())   # Cat's overridden version
print(repr(rex))      # inherited __repr__ from Animal

# The inheritance chain:
print(issubclass(Dog, Animal))   # True
print(isinstance(rex, Animal))   # True — rex IS-A Animal

## 2. Extending and Overriding the Parent

Two common patterns:
- **Override** - completely replace the parent method.
- **Extend** - call the parent method with `super()` and add new behaviour.


In [ ]:
class Vehicle:
    def __init__(self, make, model, year):
        self.make  = make
        self.model = model
        self.year  = year

    def start(self):
        return f'{self.make} {self.model} engine started.'

    def info(self):
        return f'{self.year} {self.make} {self.model}'


class ElectricVehicle(Vehicle):
    def __init__(self, make, model, year, battery_kwh):
        # 1. Delegate parent initialisation to Vehicle.
        super().__init__(make, model, year)
        # 2. Add EV-specific attribute.
        self.battery_kwh = battery_kwh

    # OVERRIDE: replace parent's start entirely.
    def start(self):
        return f'{self.make} {self.model} powered up silently.'

    # EXTEND: call parent's info and add more detail.
    def info(self):
        base = super().info()          # get what the parent says
        return f'{base} (EV, {self.battery_kwh} kWh)'


car = Vehicle('Toyota', 'Corolla', 2020)
ev  = ElectricVehicle('Tesla', 'Model 3', 2024, 75)

print(car.start())
print(ev.start())   # overridden
print(car.info())
print(ev.info())    # extended

## 3. The Use of `super()`

`super()` returns a proxy that delegates to the **next class in the MRO**. Always call `super().__init__()` in `__init__` so every class in the chain gets to initialise.


In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age
        print(f'  Person.__init__ for {name}')

    def greet(self):
        return f'Hi, I am {self.name}.'


class Employee(Person):
    def __init__(self, name, age, company):
        super().__init__(name, age)       # delegate up to Person
        self.company = company
        print(f'  Employee.__init__ for {name}')

    def greet(self):
        base = super().greet()            # get Person's greeting
        return f'{base} I work at {self.company}.'


class Manager(Employee):
    def __init__(self, name, age, company, team_size):
        super().__init__(name, age, company)  # delegate up to Employee
        self.team_size = team_size
        print(f'  Manager.__init__ — manages {team_size} people')

    def greet(self):
        base = super().greet()
        return f'{base} I manage {self.team_size} people.'


print('Creating a Manager:')
m = Manager('Alice', 35, 'Acme Corp', 8)
print()
print(m.greet())

## 4. Cooperative Multiple Inheritance

Python allows a class to inherit from **multiple parents**. The **diamond problem** arises when two parents share a common grandparent. Python solves it with the **C3 linearisation algorithm** (MRO). For `super()` to work correctly in multiple inheritance, every class must call `super().__init__()` — this is *cooperative* inheritance.


In [ ]:
class Base:
    def __init__(self):
        print('  Base.__init__')
        super().__init__()   # must call super even in Base

class A(Base):
    def __init__(self):
        print('  A.__init__')
        super().__init__()   # passes up the MRO chain

class B(Base):
    def __init__(self):
        print('  B.__init__')
        super().__init__()

class C(A, B):               # inherits from BOTH A and B
    def __init__(self):
        print('  C.__init__')
        super().__init__()


print('MRO for C:')
for cls in C.__mro__:
    print(f'  {cls.__name__}')

print()
print('Creating C() — each __init__ called exactly ONCE:')
c = C()

## 5. Mixin Classes

A **mixin** provides a bundle of methods via inheritance but is not meant to be instantiated on its own. Mixins add a specific capability to whatever class mixes them in.


In [ ]:
import json

# Mixin 1 — adds JSON serialisation.
class JSONMixin:
    def to_json(self):
        return json.dumps(vars(self), indent=2)

# Mixin 2 — adds a nice __repr__.
class ReprMixin:
    def __repr__(self):
        attrs = ', '.join(f'{k}={v!r}' for k, v in vars(self).items())
        return f'{type(self).__name__}({attrs})'

# Mixin 3 — adds comparison by a _key property.
class ComparableMixin:
    def __eq__(self, other):
        return self._key == other._key
    def __lt__(self, other):
        return self._key < other._key


# Combine multiple mixins with a real class.
class Product(JSONMixin, ReprMixin, ComparableMixin):
    def __init__(self, name, price):
        self.name  = name
        self.price = price

    @property
    def _key(self):       # required by ComparableMixin
        return self.price


apple  = Product('Apple',  0.99)
banana = Product('Banana', 0.49)
cherry = Product('Cherry', 1.99)

print(repr(apple))
print(apple.to_json())
print(f'apple < cherry: {apple < cherry}')
print('Sorted:', sorted([cherry, apple, banana]))

## 6. Attribute Lookup and the Method Resolution Order

When you access `obj.attr`, Python searches in this order:
1. The object's own `__dict__`
2. The class's `__dict__`
3. Parent classes, following the **MRO**

Inspect the MRO with `ClassName.__mro__` or `ClassName.mro()`.


In [ ]:
class A:
    def hello(self):
        return 'Hello from A'

class B(A):
    pass                    # no hello — will be found in A

class C(A):
    def hello(self):
        return 'Hello from C'   # overrides A

class D(B, C):              # which hello() does D get?
    pass


print('MRO for D:  ', [c.__name__ for c in D.__mro__])

d = D()
print(d.hello())   # 'Hello from C' — B has no hello, next in MRO is C

In [ ]:
class MyClass:
    x = 'class x'    # class attribute

    def __init__(self):
        self.y = 'instance y'


obj = MyClass()

print('Instance __dict__:', obj.__dict__)     # only instance attributes
print('Class __dict__ keys:', [k for k in MyClass.__dict__ if not k.startswith('__')])

# Lookup: instance dict first, then class dict.
print(obj.y)    # in obj.__dict__
print(obj.x)    # not in obj.__dict__ — found in MyClass.__dict__

# Shadow the class attribute on the instance.
obj.x = 'instance x'
print(obj.x)          # found in obj.__dict__ — class attr shadowed
print(MyClass.x)      # class attr unchanged

## 7. `isinstance` and `issubclass`

`isinstance(obj, cls)` - checks the full inheritance chain, not just the exact type.
`issubclass(cls, other)` — checks class hierarchy.

Both accept a **tuple** of classes to check against multiple types at once.


In [ ]:
class Shape: pass
class Polygon(Shape): pass
class Triangle(Polygon): pass

t = Triangle()

print(isinstance(t, Triangle))   # True — direct type
print(isinstance(t, Polygon))    # True — parent
print(isinstance(t, Shape))      # True — grandparent
print(isinstance(t, object))     # True — everything inherits from object
print(isinstance(t, list))       # False

# Tuple form — matches any of the given types.
print(isinstance(t, (Triangle, list, dict)))  # True

print(issubclass(Triangle, Polygon))   # True
print(issubclass(Polygon, Triangle))   # False — wrong direction

# isinstance is preferred over type() ==
class Animal: pass
class Dog(Animal): pass
fido = Dog()
print(type(fido) == Animal)     # False — only exact type match
print(isinstance(fido, Animal)) # True  — respects inheritance

## 8. Inside Python Objects - `__dict__` and Attribute Storage

Every ordinary Python object stores its instance attributes in a plain dict called `__dict__`. You can inspect and even modify it directly.


In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


p = Point(3, 4)

# __dict__ is just a regular dict.
print(p.__dict__)           # {'x': 3, 'y': 4}

# Add attributes dynamically — they land in __dict__.
p.label = 'origin'
print(p.__dict__)           # {'x': 3, 'y': 4, 'label': 'origin'}

# vars(obj) is equivalent to obj.__dict__
print(vars(p))

# You can write via __dict__ too (same as setattr).
p.__dict__['colour'] = 'red'
print(p.colour)             # 'red'

## 9. `__slots__`

Declaring `__slots__` replaces the per-instance `__dict__` with fixed named slots — saving memory and preventing typos in attribute names.


In [1]:
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ('x', 'y')   # only x and y are allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y


import sys
rp = RegularPoint(1, 2)
sp = SlottedPoint(1, 2)

print(f'RegularPoint size: {sys.getsizeof(rp)} bytes')
print(f'SlottedPoint size: {sys.getsizeof(sp)} bytes')

print('Regular has __dict__:', hasattr(rp, '__dict__'))   # True
print('Slotted has __dict__:', hasattr(sp, '__dict__'))   # False

rp.z = 99               # fine — regular objects are flexible
try:
    sp.z = 99           # error — only 'x' and 'y' allowed
except AttributeError as e:
    print(f'Slots block undeclared attr: {e}')

RegularPoint size: 48 bytes
SlottedPoint size: 48 bytes
Regular has __dict__: True
Slotted has __dict__: False
Slots block undeclared attr: 'SlottedPoint' object has no attribute 'z' and no __dict__ for setting new attributes


## 10. Class Dictionaries and `dir()`

Classes are also objects. Their attributes live in a `mappingproxy` (a read-only dict view). `dir()` gives a flattened view of all names accessible on an object, including inherited ones.


In [2]:
class Base:
    x = 1
    def base_method(self): pass

class Child(Base):
    y = 2
    def child_method(self): pass

c = Child()

# __dict__ — only what Child itself defines.
print('Child.__dict__ (public):', [k for k in Child.__dict__ if not k.startswith('__')])

# dir() — everything accessible, including inherited names.
public = [name for name in dir(c) if not name.startswith('__')]
print('dir(c) public names:', public)

# type() returns the class of an object — and type(MyClass) is type itself.
print(type(c))           # <class 'Child'>
print(type(Child))       # <class 'type'>  — the metaclass

Child.__dict__ (public): ['y', 'child_method']
dir(c) public names: ['base_method', 'child_method', 'x', 'y']
<class '__main__.Child'>
<class 'type'>


## 11. Composition over Inheritance

**Inheritance** models IS-A: `Dog IS-A Animal`.
**Composition** models HAS-A: `Car HAS-A Engine`.

Prefer composition when you need to swap behaviour at runtime, or when deep hierarchies become hard to follow.


In [ ]:
# Inheritance approach — hierarchy explodes with many capability combinations.
class FlyingSwimmingDuck:
    def fly(self):   return 'flap flap'
    def swim(self):  return 'splish splash'

# What about RubberDuck (swims, no fly)? DecoyDuck (neither)? Classes multiply.

# ── Composition approach — behaviours are separate objects ────────────────
class FlyBehaviour:
    def fly(self): return 'flap flap — I can fly!'

class NoFlyBehaviour:
    def fly(self): return 'I cannot fly.'

class SwimBehaviour:
    def swim(self): return 'splish splash!'

class FloatBehaviour:
    def swim(self): return 'I float but do not swim.'


class Duck:
    '''A Duck HAS-A fly behaviour and HAS-A swim behaviour.'''
    def __init__(self, name, fly_beh, swim_beh):
        self.name   = name
        self._fly   = fly_beh    # compose in a behaviour object
        self._swim  = swim_beh

    def fly(self):   return self._fly.fly()
    def swim(self):  return self._swim.swim()
    def quack(self): return 'Quack!'

    # Behaviour can be swapped at RUNTIME — impossible with inheritance!
    def set_fly(self, beh):
        self._fly = beh


mallard = Duck('Mallard', FlyBehaviour(),   SwimBehaviour())
rubber  = Duck('Rubber',  NoFlyBehaviour(), FloatBehaviour())

print(mallard.fly(), mallard.swim())
print(rubber.fly(),  rubber.swim())

mallard.set_fly(NoFlyBehaviour())   # clip the wings at runtime
print(mallard.fly())

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| Single inheritance | child gets all parent attributes and methods |
| Override | replace a parent method in the child |
| Extend | call `super().method()` then add your own logic |
| `super()` | proxy to next class in MRO — always use in `__init__` |
| Cooperative MI | every class calls `super().__init__()` |
| Mixins | reusable bundles of methods, not for instantiation |
| MRO (C3) | the lookup order; inspect with `__mro__` |
| `isinstance` / `issubclass` | check the full chain — prefer over `type() ==` |
| `__dict__` | per-instance attribute storage |
| `__slots__` | fixed attributes, memory savings |
| Composition | HAS-A over IS-A when flexibility matters |

